# A first neural network

This notebook goes with [docs/ml_primer.md](../docs/ml_primer.md), *Machine learning from zero*, sections 4 and 5. It
builds a tagger, a method that scores each jet, to separate W jets (from a decaying W boson) from QCD jets (from a
single quark or gluon). The tagger reads two numbers per jet
([primer](../docs/primer.md#a-first-look-at-the-observables)). The first is the soft-drop mass $m_\mathrm{SD}$, the jet
mass after soft, wide-angle particles are removed. The second is $\tau_{21}$, which is small for a jet with two prongs.
The tagger is built in small steps, from a straight line trained by hand to a network trained with PyTorch.

- **Part A** (sections 1-3) is task `W2.3.1` of [TASKS.md](../TASKS.md): read the first half of [ml_primer section
  4](../docs/ml_primer.md#neural-networks) and [section 5](../docs/ml_primer.md#pytorch-in-one-page) first.
- **Part B** (sections 4-8) is task `W2.3.2`: read the second half of ml_primer section 4 first.

Run the cells in order. If a cell ran twice and the numbers look odd, restart the kernel and run all cells again. The
notebook makes its own 6 000 jets, so it needs no `data/` folder. It runs in under a minute of CPU. Run it with the
kernel "Python (inside-the-jet)" ([docs/install.md](../docs/install.md)). Each section ends with *Try*: the changes that
the questions ask you to make and describe.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

import itj.features
import itj.metrics
import itj.toy

# The week-2 box trains the course models in the background with several cores. Two threads for this notebook keep
# both responsive; the networks here are small, so they barely notice.
torch.set_num_threads(2)
torch.manual_seed(0)

## Part A: the parts of a network, by hand

### 1. Twelve numbers per jet

The cell generates 3 000 QCD jets and 3 000 W jets at the `detector` level, which takes about 10 s.
`itj.features.jet_features` gives the 12 numbers per jet that two later taggers read: the BDT (boosted decision trees)
of `W2.2.2` and the MLP (multi-layer perceptron, a basic neural network) of `W2.3.3`. Here we keep two of them. The
soft-drop mass is divided by 100 GeV so that both inputs are numbers of order 1. `W2.3.3` does this properly for all
twelve, with standardisation (shifting and scaling each input to zero mean and unit spread). The label is 1 for a W jet
and 0 for a QCD jet. `split` gives the three parts of [ml_primer section
2](../docs/ml_primer.md#train-validation-and-test): train, validation and test. This notebook only trains and validates;
it never looks at the test part.

In [ ]:
data = itj.toy.generate(6000, processes=("qcd", "wqq"), seed=2, truth=False)
X12, names = itj.features.jet_features(data)
print(X12.shape, names)

train, val, test = itj.features.split(data)  # test = first 20 %, val = next 10 %, train = the rest


def two_inputs(part):
    """(m_SD / 100 GeV, tau21) of every jet as float32, and the label: 1 for W, 0 for QCD."""
    X, names = itj.features.jet_features(part)
    x = np.stack([X[:, names.index("jet_sdmass")] / 100.0, X[:, names.index("tau21")]], axis=1)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(part.process_names == "wqq", dtype=torch.float32)


x_train, y_train = two_inputs(train)
x_val, y_val = two_inputs(val)
print(f"train {len(x_train)} jets, val {len(x_val)} jets; fraction of W jets in train {y_train.mean():.2f}")

*Try:* plot the two inputs of the train part against each other, coloured by the label. Where would you draw a line?

### 2. Tensors and autograd

A tensor is PyTorch's array, and autograd is its automatic differentiation. A tensor made with `requires_grad=True`
records what is done with it, and `.backward()` then computes the derivative. Here we take a two-line function,
$f(a) = a^3 - 2a$, at $a = 2$, and compare the derivative from autograd with the derivative by hand, $f'(a) = 3a^2 - 2$.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
f = a**3 - 2 * a
f.backward()  # df/da, stored in a.grad
print(f"f = {f.item()}, autograd df/da = {a.grad.item()}, by hand 3a^2 - 2 = {3 * 2.0**2 - 2}")

# The same works for many numbers at once: the gradient of a sum has one entry per input.
v = torch.tensor([0.0, 1.0, 2.0], requires_grad=True)
(v**3 - 2 * v).sum().backward()
print("gradient of the sum:", v.grad)

*Try:* compare autograd with the derivative by hand for another function of your choice, for example $\sin(a)\,e^{a}$.

### 3. Logistic regression, written by hand

Logistic regression is the model of [ml_primer section 4](../docs/ml_primer.md#neural-networks): a linear function of
the inputs passed through the sigmoid function, which turns it into a probability, $p = \sigma(w \cdot x + b)$. The
loss, the number that training makes small, is the cross-entropy. Gradient descent moves $w$ and $b$ in small steps
against the gradient of the loss, $\partial L/\partial w = \frac{1}{N}\sum_i (p_i - y_i)\,x_i$ and
$\partial L/\partial b = \frac{1}{N}\sum_i (p_i - y_i)$. This part uses tensors and arithmetic only, without the PyTorch
tools of part B. `n_steps` and `eta` (the learning rate, the size of each step) are the two settings to vary.

In [ ]:
n_steps, eta = 300, 1.0
w, b = torch.zeros(2), torch.tensor(0.0)
losses = []
for _ in range(n_steps):  # one gradient step per round
    p = torch.sigmoid(x_train @ w + b)  # the probability of "W" for every train jet
    loss = -(y_train * torch.log(p) + (1 - y_train) * torch.log(1 - p)).mean()
    losses.append(loss.item())
    grad_w = ((p - y_train)[:, None] * x_train).mean(dim=0)
    grad_b = (p - y_train).mean()
    w, b = w - eta * grad_w, b - eta * grad_b  # one step of gradient descent

print(f"w = {w.numpy().round(3)}, b = {b.item():.3f}, final train loss {losses[-1]:.4f}")
score_val = (x_val @ w + b).numpy()  # the sigmoid does not change the order, so z is a valid score
print(f"AUC on the val part: {itj.metrics.auc(y_val.numpy(), score_val):.3f}")

fig, ax = plt.subplots(figsize=(6.0, 3.6), layout="constrained")
ax.plot(losses)
ax.set_xlabel("gradient step")
ax.set_ylabel("cross-entropy (train)")
ax.set_title(f"logistic regression by hand, learning rate {eta}")
plt.show()

*Try (`W2.3.1`):* change `n_steps` (30, 3 000) and `eta` (0.1, 10, 50). Describe what happens to the loss curve and to
the AUC, the area under the ROC curve ([ml_primer](../docs/ml_primer.md#a-tagger-is-a-function)). Save your figure of
the loss against the step as `W2.3.1_logistic.pdf` (`fig.savefig(...)`). What can a single straight line in the plane of
$(m_\mathrm{SD}, \tau_{21})$ do, and what can it not?

## Part B: a network and its training loop

### 4. The same model as an `nn.Module`

This is the logistic regression of part A again, now written the way PyTorch is normally used. The model is an
`nn.Module` that holds its parameters. Autograd computes the gradients, and an optimiser (the rule that updates the
weights) takes the step. The three lines `zero_grad`, `backward` and `step` are the core of every training loop.
`BCEWithLogitsLoss` is the cross-entropy of part A, computed from $z$ (the *logit*, the value before the sigmoid) in a
numerically safer way.

In [ ]:
class Logistic(nn.Module):
    """The logistic regression of part A: z = w . x + b."""

    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)  # w and b

    def forward(self, x):
        """Return the logit z of every jet."""
        return self.linear(x).squeeze(-1)  # z, one number per jet


model = Logistic(2)
nn.init.zeros_(model.linear.weight), nn.init.zeros_(model.linear.bias)  # start where part A started
opt = torch.optim.SGD(model.parameters(), lr=1.0)  # plain gradient descent, as in part A
loss_fn = nn.BCEWithLogitsLoss()
for _ in range(300):
    loss = loss_fn(model(x_train), y_train)
    opt.zero_grad()  # forget the gradients of the previous step
    loss.backward()  # compute the new ones
    opt.step()  # move the weights
print(model)
print("weights", model.linear.weight.data.numpy().round(3), "bias", model.linear.bias.data.numpy().round(3))
print(f"final train loss {loss.item():.4f} (part A: {losses[-1]:.4f})")

*Try:* compare the weights with those of part A. Do they agree, and why?

### 5. An MLP with `nn.Sequential`

The MLP has three linear layers with ReLU activations between them (a ReLU sets negative values to zero). Its two hidden
layers have 32 units each. `print(model)` shows the layers, and the `state_dict` holds the weights by name.

In [ ]:
def make_mlp(width=32, depth=2, n_inputs=2):
    """Build an MLP: depth hidden layers of the given width, a ReLU after each, one output (the logit z)."""
    layers, size = [], n_inputs
    for _ in range(depth):
        layers += [nn.Linear(size, width), nn.ReLU()]
        size = width
    return nn.Sequential(*layers, nn.Linear(size, 1))


mlp = make_mlp()
print(mlp)
print({name: tuple(t.shape) for name, t in mlp.state_dict().items()})
print("parameters:", sum(t.numel() for t in mlp.parameters()))

*Try:* how many parameters does a width of 256 give? Count them by hand first.

### 6. Mini-batches, AdamW and a validation loss after every epoch

This is the training loop of every network in this course. It shuffles the train jets and walks through them in
mini-batches (small groups of jets). It takes one step of AdamW, the optimiser of the course, per batch. After each
epoch (one pass over the train jets) it computes the loss of the train and of the val part in evaluation mode, without
gradients. It has no early stopping and no standardisation: `W2.3.3` adds both to your own copy of it.

In [ ]:
def train_model(model, x_train, y_train, x_val, y_val, lr=1e-3, batch_size=64, epochs=30, seed=0):
    """Train with mini-batches and AdamW; return the train and val loss after every epoch."""
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    history = {"train": [], "val": []}
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(x_train))  # a new order of the jets in every epoch
        for start in range(0, len(x_train), batch_size):
            idx = order[start : start + batch_size]
            loss = loss_fn(model(x_train[idx]).squeeze(-1), y_train[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(x_train).squeeze(-1), y_train).item())
            history["val"].append(loss_fn(model(x_val).squeeze(-1), y_val).item())
    return history


def plot_history(histories, title):
    """Draw the train (dashed) and val (solid) loss of every run against the epoch."""
    fig, ax = plt.subplots(figsize=(6.0, 3.6), layout="constrained")
    for k, (label, history) in enumerate(histories.items()):
        ax.plot(history["train"], color=f"C{k}", ls="--", label=f"{label}, train")
        ax.plot(history["val"], color=f"C{k}", label=f"{label}, val")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cross-entropy")
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.show()
    return fig


mlp = make_mlp()
history = train_model(mlp, x_train, y_train, x_val, y_val)  # a few seconds
plot_history({"MLP 2-32-32-1": history}, f"{len(x_train)} train jets, batch 64, AdamW, learning rate 1e-3")
with torch.no_grad():
    print(f"AUC on the val part: {itj.metrics.auc(y_val.numpy(), mlp(x_val).squeeze(-1).numpy()):.3f}")

*Try:* compare the AUC with that of the straight line of section 3. What can the MLP do that the line could not?

### 7. Overfitting

Overfitting means that a model learns its training set by heart and does worse on new data. Here a large network (three
hidden layers of 256 units) is trained on only 500 train jets for 300 epochs, which takes about 10 s. Watch the two
curves, the loss of the train part and of the val part.

In [ ]:
big = make_mlp(width=256, depth=3)
history_big = train_model(big, x_train[:500], y_train[:500], x_val, y_val, lr=3e-3, batch_size=50, epochs=300)
plot_history({"MLP 2-256-256-256-1, 500 jets": history_big}, "a large network on few jets")
best = int(np.argmin(history_big["val"]))
print(f"lowest val loss {history_big['val'][best]:.3f} after epoch {best + 1}; at the end {history_big['val'][-1]:.3f}")

*Try:* in one sentence, what happens after the lowest point of the val loss, and why? Which epoch's weights would you
keep? Then train the same network on all train jets (fewer epochs): does it still overfit?

### 8. Change one thing

The cell trains the MLP of section 6 at three learning rates. Change one thing at a time, keep everything else fixed,
and look at both curves.

In [ ]:
histories = {}
for lr in (1e-2, 1e-3, 1e-4):
    histories[f"lr {lr:g}"] = train_model(make_mlp(), x_train, y_train, x_val, y_val, lr=lr)
fig = plot_history(histories, "the MLP of section 6 at three learning rates")

*Try (`W2.3.2`):* describe what changes when the learning rate is multiplied and divided by 10, and save the figure as
`W2.3.2_loss.pdf`. Then vary the batch size: 16 and 512 at the learning rate 1e-3. Which settings would you choose, and
on which part of the data did you choose them?

**Next:** `W2.3.3` starts from a copy of `train_model` (copy it, do not import it). It uses twelve standardised inputs
instead of two, and early stopping (the training ends when the val loss stops falling) with the best weights restored.